# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [24]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [25]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [26]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [27]:

EVENT_NAME = '202309_Earthquake_Morocco'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'aria'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [28]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [29]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 3 .tif files in the S3 bucket.


['drcs_activations/202309_Earthquake_Morocco/aria/ARIA_DPM_ALOS2_Morocco_earthquake.tif',
 'drcs_activations/202309_Earthquake_Morocco/aria/MOROCCO_S1_20230830_20230911_UNW.tif',
 'drcs_activations/202309_Earthquake_Morocco/aria/MOROCCO_S1_20230830_20230911_WRP.tif']

## Configure bucket and paths (no need to create session manually)

In [30]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [31]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 384
  - Total size: 77.34 GB

📁 Cached files (first 10):
  - drcs_activations/20230719_SevereWx_NC/aria/ARIA_DPM_Sentinel-1_North_Carolina_Tornado.tif (4.2 MB)
  - drcs_activations/20230719_SevereWx_NC/aria/ARIA_DPMraw_Sentinel-1_North_Carolina_Tornado.tif (4.2 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_colorInfrared.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_naturalColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_trueColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_colorInfrared.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_naturalColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_trueColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/sentinel1

(384, 83038096604)

In [32]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [33]:
keys

['drcs_activations/202309_Earthquake_Morocco/aria/ARIA_DPM_ALOS2_Morocco_earthquake.tif',
 'drcs_activations/202309_Earthquake_Morocco/aria/MOROCCO_S1_20230830_20230911_UNW.tif',
 'drcs_activations/202309_Earthquake_Morocco/aria/MOROCCO_S1_20230830_20230911_WRP.tif']

In [37]:
# Define filename creator functions for different file types

def create_cog_filename_simple_prefix(f, EVENT_NAME):
    """Move EVENT_NAME to beginning and extract YYYYMM to end of filename."""
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Extract date from EVENT_NAME (format: YYYYMM_EventType_Location)
    event_parts = EVENT_NAME.split('_')
    if event_parts and len(event_parts[0]) >= 6 and event_parts[0][:6].isdigit():
        year_month = event_parts[0][:6]  # Extract YYYYMM
        formatted_date = f"{year_month[:4]}{year_month[4:6]}"  # Format as YYYY-MM
        
        # Create new filename: EVENT_NAME_original_filename_YYYY-MM.tif
        cog_filename = f'{EVENT_NAME}_{filename}_{formatted_date}month{extension}'
    else:
        # Fallback if EVENT_NAME doesn't have expected format
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = 'ARIA_DPM_ALOS2'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_simple_prefix(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202309_Earthquake_Morocco_ARIA_DPM_ALOS2_Morocco_earthquake_202309month.tif


In [38]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_simple_prefix, 
                                target_dir = "ALOS2", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202309_Earthquake_Morocco_ARIA_DPM_ALOS2_Morocco_earthquake_202309month.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202309_Earthquake_Morocco/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ALOS2

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202309_Earthquake_Morocco

[1/1] Processing: drcs_activations/202309_Earthquake_Morocco/aria/ARIA_DPM_ALOS2_Morocco_earthquake.tif
   Output filename: 202309_Earthquake_Morocco_ARIA_DPM_ALOS2_Morocco_earthquake_202309month.tif
   [MEMORY] Initial: 290.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=78820/1000000
            Estimated data coverage: 11.8% (from distributed samples)
   [VERIFY] Band 2: min=0, max=255, center sample non-zero=72234/100000

Reading input: /tmp/tmpj2tk26q3_temp.tif



   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpe5_z2oyw.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ALOS2/202309_Earthquake_Morocco_ARIA_DPM_ALOS2_Morocco_earthquake_202309month.tif
   [MEMORY] Final: 451.9 MB (Change: +161.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Earthquake_Morocco_ARIA_DPM_ALOS2_Morocco_earthquake_202309month.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/ALOS2/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/ALOS2/files_converted.csv
📁 COGs saved locally to: output/202309_Earthquake_Morocco

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T02:20:11.248738


In [39]:
keys

['drcs_activations/202309_Earthquake_Morocco/aria/ARIA_DPM_ALOS2_Morocco_earthquake.tif',
 'drcs_activations/202309_Earthquake_Morocco/aria/MOROCCO_S1_20230830_20230911_UNW.tif',
 'drcs_activations/202309_Earthquake_Morocco/aria/MOROCCO_S1_20230830_20230911_WRP.tif']

In [47]:
# Define filename creator functions for different file types

def create_cog_filename_morocco_dates(f, EVENT_NAME):
    """Extract start and end dates from filename and format as EVENT_NAME_..._YYYYMMDDday_YYYYMMDDday.tif"""
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Extract the two dates from the filename (format: YYYYMMDD_YYYYMMDD)
    import re
    date_pattern = r'_(\d{8})_(\d{8})_'
    match = re.search(date_pattern, filename)
    
    if match:
        start_date = match.group(1)  # 20230830
        end_date = match.group(2)    # 20230911
        
        # Format dates as YYYYMMDDday
        start_formatted = f"{start_date}day"
        end_formatted = f"{end_date}"
        
        # Remove the date portion from the original filename
        filename_without_dates = re.sub(date_pattern, '_', filename)
        
        # Check if EVENT_NAME contains "Morocco" to avoid duplication
        if "Morocco" in EVENT_NAME and "MOROCCO" in filename_without_dates:
            # Remove MOROCCO from the filename part
            filename_without_dates = filename_without_dates.replace("MOROCCO_", "")
        
        # Create new filename: EVENT_NAME_remaining_parts_YYYYMMDDday_YYYYMMDDday.tif
        cog_filename = f'{EVENT_NAME}_{filename_without_dates}_{start_formatted}_{end_formatted}{extension}'
    else:
        # Fallback if dates not found in expected format
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = 'MOROCCO_S1'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_morocco_dates(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202309_Earthquake_Morocco_S1_UNW_20230830day_20230911.tif
  202309_Earthquake_Morocco_S1_WRP_20230830day_20230911.tif


In [ ]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_morocco_dates, 
                                target_dir = "Sentinel-1/UNW", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202309_Earthquake_Morocco_S1_UNW_20230830day_20230911.tif
  202309_Earthquake_Morocco_S1_WRP_20230830day_20230911.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202309_Earthquake_Morocco/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/UNW

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202309_Earthquake_Morocco

[1/2] Processing: drcs_activations/202309_Earthquake_Morocco/aria/MOROCCO_S1_20230830_20230911_UNW.tif
   Output filename: 202309_Earthquake_Morocco_S1_UNW_20230830day_20230911.tif
   [MEMORY] Initial: 452.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection


Reading input: /tmp/tmppygfdsut_temp.tif



   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-0.047381024807691574, max=0.03953872248530388, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp6mtjd2af.tif


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")